# Step 4 Gom cụm Wine Quality bằng K-Means và DBSCAN

Notebook này được tách thành từng phần nhỏ để dễ trình bày. Mỗi phần có mô tả công dụng, sau đó là đoạn code tương ứng. Chạy lần lượt từ trên xuống hoặc chọn **Run All** để tạo lại báo cáo HTML/PDF.


## 1. Khai báo thư viện và tham số

**Công dụng:** Nạp các thư viện chuẩn của Python, khai báo đường dẫn file dữ liệu, tên file báo cáo, tham số K-Means/DBSCAN và bảng màu dùng cho biểu đồ.


In [ ]:
print('Công dụng: Khai báo thư viện, file dữ liệu, tham số và bảng màu.')

from collections import Counter, defaultdict
from html import escape
from math import comb, log, sqrt
from pathlib import Path
from random import Random
from zipfile import ZipFile
from xml.etree import ElementTree as ET


BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_FILE_NAME = "D09 - winequality-red.xlsx"

def find_project_dir(start_dir):
    for directory in [start_dir, *start_dir.parents]:
        if (directory / "data" / DATA_FILE_NAME).exists():
            return directory
    raise FileNotFoundError(f"Khong tim thay data/{DATA_FILE_NAME}")

PROJECT_DIR = find_project_dir(BASE_DIR)
NOTEBOOK_DIR = BASE_DIR if (BASE_DIR / "main.ipynb").exists() else PROJECT_DIR / "notebook" 
DATA_FILE = PROJECT_DIR / "data" / DATA_FILE_NAME
OUTPUT_DIR = PROJECT_DIR / "result"
VISUAL_REPORT = OUTPUT_DIR / "clustering_report.html"
RANDOM_STATE = 42
DBSCAN_EPS = 1.3
DBSCAN_MIN_SAMPLES = 5
PALETTE = [
    "#2563eb",
    "#dc2626",
    "#16a34a",
    "#9333ea",
    "#ea580c",
    "#0891b2",
    "#be123c",
    "#4f46e5",
    "#65a30d",
    "#a16207",
    "#0f766e",
    "#7c3aed",
]


## 2. Đọc dữ liệu Excel

**Công dụng:** Đọc file `.xlsx` bằng thư viện chuẩn, tự tìm sheet chứa dữ liệu nhiều dòng nhất, lấy 11 cột đặc trưng vào `X` và cột nhãn cuối `quality` vào `y`. Cột `quality` không được đưa vào dữ liệu gom cụm.


In [ ]:
print('Công dụng: Đọc file Excel, lấy đặc trưng X và nhãn thật y.')

def _column_index(cell_ref):
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    index = 0
    for ch in letters:
        index = index * 26 + ord(ch.upper()) - ord("A") + 1
    return index - 1

def _cell_value(cell, shared_strings, ns):
    value_node = cell.find("a:v", ns)
    if value_node is None:
        return ""

    value = value_node.text or ""
    if cell.get("t") == "s":
        return shared_strings[int(value)]
    return value

def read_xlsx_sheet(path):
    """Read the worksheet that contains the tabular wine data."""
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

    with ZipFile(path) as workbook:
        shared_strings = []
        if "xl/sharedStrings.xml" in workbook.namelist():
            root = ET.fromstring(workbook.read("xl/sharedStrings.xml"))
            for si in root.findall("a:si", ns):
                text = "".join(t.text or "" for t in si.findall(".//a:t", ns))
                shared_strings.append(text)

        sheet_names = sorted(
            name
            for name in workbook.namelist()
            if name.startswith("xl/worksheets/sheet") and name.endswith(".xml")
        )

        best_rows = []
        for sheet_name in sheet_names:
            root = ET.fromstring(workbook.read(sheet_name))
            rows = []
            for row_node in root.findall(".//a:sheetData/a:row", ns):
                row = []
                for cell in row_node.findall("a:c", ns):
                    col = _column_index(cell.get("r", "A1"))
                    while len(row) <= col:
                        row.append("")
                    row[col] = _cell_value(cell, shared_strings, ns)
                rows.append(row)

            if len(rows) > len(best_rows):
                best_rows = rows

    if not best_rows:
        raise ValueError("Khong tim thay du lieu trong file Excel.")

    header = best_rows[0]
    data_rows = best_rows[1:]
    return header, data_rows

def load_wine_quality(path):
    header, rows = read_xlsx_sheet(path)
    label_name = header[-1]
    feature_names = header[:-1]

    X = []
    y = []
    for row in rows:
        if len(row) < len(header) or any(value == "" for value in row[: len(header)]):
            continue
        X.append([float(value) for value in row[:-1]])
        y.append(int(float(row[-1])))

    return feature_names, label_name, X, y


## 3. Chuẩn hóa dữ liệu

**Công dụng:** Đưa các đặc trưng về cùng thang đo bằng Z-score. Việc này quan trọng vì K-Means và DBSCAN đều dựa trên khoảng cách.


In [ ]:
print('Công dụng: Chuẩn hóa các đặc trưng bằng Z-score.')

def standardize(X):
    n_samples = len(X)
    n_features = len(X[0])
    means = [sum(row[j] for row in X) / n_samples for j in range(n_features)]
    stds = []

    for j in range(n_features):
        variance = sum((row[j] - means[j]) ** 2 for row in X) / n_samples
        stds.append(sqrt(variance) or 1.0)

    return [
        [(row[j] - means[j]) / stds[j] for j in range(n_features)]
        for row in X
    ]

def squared_distance(a, b):
    return sum((x - y) ** 2 for x, y in zip(a, b))


## 4. Thuật toán K-Means

**Công dụng:** Chia dữ liệu thành `k` cụm. Trong bài này `k = 6` vì nhãn thật `quality` có 6 mức: `3, 4, 5, 6, 7, 8`. K-Means cần biết trước số cụm.


In [ ]:
print('Công dụng: Định nghĩa thuật toán K-Means.')

def kmeans(X, n_clusters, max_iter=300, random_state=RANDOM_STATE):
    rng = Random(random_state)
    centroids = [X[i][:] for i in rng.sample(range(len(X)), n_clusters)]
    labels = [-1] * len(X)

    for _ in range(max_iter):
        new_labels = [
            min(range(n_clusters), key=lambda k: squared_distance(row, centroids[k]))
            for row in X
        ]

        if new_labels == labels:
            break

        labels = new_labels
        grouped = defaultdict(list)
        for label, row in zip(labels, X):
            grouped[label].append(row)

        for k in range(n_clusters):
            if not grouped[k]:
                centroids[k] = X[rng.randrange(len(X))][:]
                continue

            centroids[k] = [
                sum(row[j] for row in grouped[k]) / len(grouped[k])
                for j in range(len(X[0]))
            ]

    return labels


## 5. Thuật toán DBSCAN

**Công dụng:** Gom cụm theo mật độ điểm. DBSCAN không cần nhập trước số cụm; thuật toán tự phát hiện cụm và đánh dấu các điểm lẻ là `Nhiễu` (`-1`).


In [ ]:
print('Công dụng: Định nghĩa thuật toán DBSCAN.')

def dbscan(X, eps=1.9, min_samples=12):
    labels = [None] * len(X)
    cluster_id = 0
    eps_squared = eps * eps

    neighborhoods = []
    for point in X:
        neighborhoods.append(
            [i for i, other in enumerate(X) if squared_distance(point, other) <= eps_squared]
        )

    for i in range(len(X)):
        if labels[i] is not None:
            continue

        neighbors = neighborhoods[i]
        if len(neighbors) < min_samples:
            labels[i] = -1
            continue

        labels[i] = cluster_id
        seeds = list(neighbors)
        seed_set = set(seeds)
        cursor = 0

        while cursor < len(seeds):
            point_index = seeds[cursor]
            cursor += 1

            if labels[point_index] == -1:
                labels[point_index] = cluster_id
            if labels[point_index] is not None:
                continue

            labels[point_index] = cluster_id
            point_neighbors = neighborhoods[point_index]
            if len(point_neighbors) >= min_samples:
                for neighbor in point_neighbors:
                    if neighbor not in seed_set:
                        seeds.append(neighbor)
                        seed_set.add(neighbor)

        cluster_id += 1

    return labels


## 6. Đánh giá kết quả gom cụm

**Công dụng:** So sánh cụm dự đoán với nhãn thật `quality` bằng các chỉ số `ARI`, `NMI`, `Purity`. Các chỉ số này giúp biết kết quả gom cụm có gần với nhãn chất lượng thật hay không.


In [ ]:
print('Công dụng: Tính ARI, NMI, Purity và in kết quả đánh giá.')

def adjusted_rand_index(y_true, y_pred):
    contingency = Counter(zip(y_true, y_pred))
    true_counts = Counter(y_true)
    pred_counts = Counter(y_pred)
    n = len(y_true)

    sum_comb = sum(comb(count, 2) for count in contingency.values())
    true_comb = sum(comb(count, 2) for count in true_counts.values())
    pred_comb = sum(comb(count, 2) for count in pred_counts.values())
    total_comb = comb(n, 2)

    expected = true_comb * pred_comb / total_comb if total_comb else 0.0
    max_index = (true_comb + pred_comb) / 2
    denominator = max_index - expected
    return 0.0 if denominator == 0 else (sum_comb - expected) / denominator

def normalized_mutual_information(y_true, y_pred):
    n = len(y_true)
    true_counts = Counter(y_true)
    pred_counts = Counter(y_pred)
    contingency = Counter(zip(y_true, y_pred))

    mutual_info = 0.0
    for (true_label, pred_label), count in contingency.items():
        mutual_info += (
            (count / n)
            * log((count * n) / (true_counts[true_label] * pred_counts[pred_label]))
        )

    true_entropy = -sum((count / n) * log(count / n) for count in true_counts.values())
    pred_entropy = -sum((count / n) * log(count / n) for count in pred_counts.values())
    denominator = (true_entropy + pred_entropy) / 2
    return 0.0 if denominator == 0 else mutual_info / denominator

def purity_score(y_true, y_pred):
    clusters = defaultdict(list)
    for true_label, pred_label in zip(y_true, y_pred):
        clusters[pred_label].append(true_label)

    correct = sum(Counter(labels).most_common(1)[0][1] for labels in clusters.values())
    return correct / len(y_true)

def evaluate_clustering(y_true, y_pred):
    non_noise_clusters = sorted(label for label in set(y_pred) if label != -1)
    noise_count = sum(1 for label in y_pred if label == -1)

    return {
        "clusters": len(non_noise_clusters),
        "noise": noise_count,
        "ARI": adjusted_rand_index(y_true, y_pred),
        "NMI": normalized_mutual_information(y_true, y_pred),
        "Purity": purity_score(y_true, y_pred),
    }

def print_metrics(name, metrics):
    print(name)
    print(f"  So cum: {metrics['clusters']}")
    print(f"  Diem nhieu: {metrics['noise']}")
    print(f"  ARI: {metrics['ARI']:.4f}")
    print(f"  NMI: {metrics['NMI']:.4f}")
    print(f"  Purity: {metrics['Purity']:.4f}")


## 7. PCA để vẽ biểu đồ 2 chiều

**Công dụng:** Dữ liệu có 11 đặc trưng nên khó vẽ trực tiếp. PCA chiếu dữ liệu xuống 2 chiều `PC1`, `PC2` để quan sát phân bố nhãn và cụm trên mặt phẳng.


In [ ]:
print('Công dụng: Tính PCA 2 chiều để trực quan hóa dữ liệu.')

def matrix_vector_multiply(matrix, vector):
    return [sum(value * vector[j] for j, value in enumerate(row)) for row in matrix]

def vector_norm(vector):
    return sqrt(sum(value * value for value in vector)) or 1.0

def covariance_matrix(X):
    n_samples = len(X)
    n_features = len(X[0])
    return [
        [
            sum(row[i] * row[j] for row in X) / (n_samples - 1)
            for j in range(n_features)
        ]
        for i in range(n_features)
    ]

def first_eigenpair(matrix, random_state=RANDOM_STATE, iterations=100):
    rng = Random(random_state)
    vector = [rng.random() for _ in matrix]
    norm = vector_norm(vector)
    vector = [value / norm for value in vector]

    for _ in range(iterations):
        next_vector = matrix_vector_multiply(matrix, vector)
        norm = vector_norm(next_vector)
        vector = [value / norm for value in next_vector]

    multiplied = matrix_vector_multiply(matrix, vector)
    eigenvalue = sum(a * b for a, b in zip(vector, multiplied))
    return eigenvalue, vector

def deflate_matrix(matrix, eigenvalue, eigenvector):
    return [
        [
            value - eigenvalue * eigenvector[i] * eigenvector[j]
            for j, value in enumerate(row)
        ]
        for i, row in enumerate(matrix)
    ]

def pca_2d(X):
    covariance = covariance_matrix(X)
    first_value, first_vector = first_eigenpair(covariance, RANDOM_STATE)
    reduced = deflate_matrix(covariance, first_value, first_vector)
    second_value, second_vector = first_eigenpair(reduced, RANDOM_STATE + 1)

    projected = [
        (
            sum(value * first_vector[j] for j, value in enumerate(row)),
            sum(value * second_vector[j] for j, value in enumerate(row)),
        )
        for row in X
    ]
    total_variance = sum(covariance[i][i] for i in range(len(covariance))) or 1.0
    explained = [
        max(0.0, first_value) / total_variance,
        max(0.0, second_value) / total_variance,
    ]
    return projected, explained


## 8. Tạo biểu đồ SVG và bảng mô tả

**Công dụng:** Tạo các biểu đồ phân tán, biểu đồ cột, bảng metric, bảng chéo và bảng mô tả từng cụm. Các hàm này sinh trực tiếp mã HTML/SVG, không cần thư viện ngoài như matplotlib.


In [ ]:
print('Công dụng: Tạo biểu đồ, bảng số liệu và báo cáo HTML.')

def color_for_label(label, ordered_labels):
    if label == -1:
        return "#6b7280"
    return PALETTE[ordered_labels.index(label) % len(PALETTE)]

def scale_values(values, low, high):
    min_value = min(values)
    max_value = max(values)
    if min_value == max_value:
        return [(low + high) / 2 for _ in values]
    return [
        low + (value - min_value) * (high - low) / (max_value - min_value)
        for value in values
    ]

def scatter_svg(title, x_values, y_values, labels, x_label, y_label):
    width = 560
    height = 430
    left = 62
    right = 28
    top = 42
    bottom = 62
    plot_width = width - left - right
    plot_height = height - top - bottom

    xs = scale_values(x_values, left, left + plot_width)
    ys = scale_values(y_values, top + plot_height, top)
    ordered_labels = sorted(set(labels))

    points = []
    for x, y, label, raw_x, raw_y in zip(xs, ys, labels, x_values, y_values):
        color = color_for_label(label, ordered_labels)
        points.append(
            f'<circle cx="{x:.2f}" cy="{y:.2f}" r="3" fill="{color}" '
            f'fill-opacity="0.72"><title>{escape(str(label))}: '
            f'{escape(x_label)}={raw_x:.3f}, {escape(y_label)}={raw_y:.3f}'
            f"</title></circle>"
        )

    legend_items = []
    for index, label in enumerate(ordered_labels):
        x = left + (index % 6) * 76
        y = height - 24 + (index // 6) * 18
        color = color_for_label(label, ordered_labels)
        legend_items.append(
            f'<circle cx="{x}" cy="{y}" r="5" fill="{color}"></circle>'
            f'<text x="{x + 9}" y="{y + 4}" class="legend">{escape(str(label))}</text>'
        )

    return f"""
    <svg viewBox="0 0 {width} {height}" role="img" aria-label="{escape(title)}">
      <text x="{left}" y="24" class="chart-title">{escape(title)}</text>
      <line x1="{left}" y1="{top + plot_height}" x2="{left + plot_width}" y2="{top + plot_height}" class="axis"></line>
      <line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_height}" class="axis"></line>
      <text x="{left + plot_width / 2}" y="{height - 34}" text-anchor="middle" class="axis-label">{escape(x_label)}</text>
      <text x="18" y="{top + plot_height / 2}" text-anchor="middle" transform="rotate(-90 18 {top + plot_height / 2})" class="axis-label">{escape(y_label)}</text>
      {''.join(points)}
      {''.join(legend_items)}
    </svg>
    """

def metrics_bar_svg(kmeans_metrics, dbscan_metrics):
    width = 720
    height = 320
    left = 82
    top = 34
    max_bar = 460
    row_gap = 70
    metrics = ["ARI", "NMI", "Purity"]
    bars = []

    for row, metric in enumerate(metrics):
        y = top + row * row_gap
        bars.append(f'<text x="0" y="{y + 24}" class="metric-name">{metric}</text>')

        k_width = max(0.0, kmeans_metrics[metric]) * max_bar
        d_width = max(0.0, dbscan_metrics[metric]) * max_bar

        bars.append(
            f'<rect x="{left}" y="{y}" width="{k_width:.2f}" height="22" rx="4" fill="#2563eb"></rect>'
            f'<text x="{left + k_width + 8:.2f}" y="{y + 16}" class="value">{kmeans_metrics[metric]:.4f}</text>'
            f'<rect x="{left}" y="{y + 28}" width="{d_width:.2f}" height="22" rx="4" fill="#dc2626"></rect>'
            f'<text x="{left + d_width + 8:.2f}" y="{y + 44}" class="value">{dbscan_metrics[metric]:.4f}</text>'
        )

    return f"""
    <svg viewBox="0 0 {width} {height}" role="img" aria-label="So sánh chỉ số đánh giá">
      <text x="0" y="20" class="chart-title">So sánh chỉ số đánh giá</text>
      <line x1="{left}" y1="{height - 46}" x2="{left + max_bar}" y2="{height - 46}" class="axis"></line>
      <text x="{left}" y="{height - 24}" class="legend">0</text>
      <text x="{left + max_bar - 8}" y="{height - 24}" class="legend">1</text>
      {''.join(bars)}
      <circle cx="585" cy="58" r="6" fill="#2563eb"></circle>
      <text x="598" y="63" class="legend">K-Means</text>
      <circle cx="585" cy="84" r="6" fill="#dc2626"></circle>
      <text x="598" y="89" class="legend">DBSCAN</text>
    </svg>
    """

def count_bar_svg(title, counts, total):
    width = 720
    height = max(260, 66 + len(counts) * 34)
    left = 92
    right = 90
    top = 42
    row_height = 28
    max_bar = width - left - right
    max_count = max(counts.values()) or 1
    rows = []

    for index, (label, count) in enumerate(counts.items()):
        y = top + index * row_height
        bar_width = count * max_bar / max_count
        percent = count * 100 / total
        label_text = "Nhiễu" if label == -1 else str(label)
        color = "#6b7280" if label == -1 else PALETTE[index % len(PALETTE)]
        rows.append(
            f'<text x="0" y="{y + 17}" class="metric-name">{escape(label_text)}</text>'
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="18" rx="4" fill="{color}"></rect>'
            f'<text x="{left + bar_width + 8:.2f}" y="{y + 14}" class="value">{count} ({percent:.1f}%)</text>'
        )

    return f"""
    <svg viewBox="0 0 {width} {height}" role="img" aria-label="{escape(title)}">
      <text x="0" y="22" class="chart-title">{escape(title)}</text>
      {''.join(rows)}
    </svg>
    """

def cluster_quality_summary(y_true, y_pred):
    summary = {}
    for pred_label in sorted(set(y_pred)):
        values = [true for true, pred in zip(y_true, y_pred) if pred == pred_label]
        counter = Counter(values)
        main_label, main_count = counter.most_common(1)[0]
        summary[pred_label] = {
            "size": len(values),
            "main_quality": main_label,
            "main_count": main_count,
            "purity": main_count / len(values),
        }
    return summary

def heat_color(value, max_value):
    if value == 0:
        return "#f8fafc", "#64748b"
    ratio = value / max_value
    red = round(239 - ratio * 209)
    green = round(246 - ratio * 182)
    blue = round(255 - ratio * 80)
    text = "#ffffff" if ratio > 0.48 else "#111827"
    return f"rgb({red}, {green}, {blue})", text

def contingency_table_html(title, y_true, y_pred):
    true_labels = sorted(set(y_true))
    pred_labels = sorted(set(y_pred))
    counts = Counter(zip(y_true, y_pred))
    max_value = max(counts.values()) or 1

    header_cells = "".join(
        f"<th>{'Nhiễu' if label == -1 else escape(str(label))}</th>"
        for label in pred_labels
    )
    rows = []

    for true_label in true_labels:
        cells = []
        for pred_label in pred_labels:
            value = counts[(true_label, pred_label)]
            background, text_color = heat_color(value, max_value)
            cells.append(
                f'<td style="background:{background};color:{text_color}">{value}</td>'
            )
        rows.append(
            f"<tr><th>quality {escape(str(true_label))}</th>{''.join(cells)}</tr>"
        )

    return f"""
    <h2>{escape(title)}</h2>
    <div class="table-scroll">
      <table class="heatmap">
        <thead><tr><th>Nhãn thật / Cụm</th>{header_cells}</tr></thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </div>
    """

def metric_table_html(kmeans_metrics, dbscan_metrics):
    rows = []
    for name, metrics in [("K-Means", kmeans_metrics), ("DBSCAN", dbscan_metrics)]:
        rows.append(
            "<tr>"
            f"<th>{name}</th>"
            f"<td>{metrics['clusters']}</td>"
            f"<td>{metrics['noise']}</td>"
            f"<td>{metrics['ARI']:.4f}</td>"
            f"<td>{metrics['NMI']:.4f}</td>"
            f"<td>{metrics['Purity']:.4f}</td>"
            "</tr>"
        )

    return f"""
    <table class="metrics-table">
      <thead>
        <tr>
          <th>Thuật toán</th>
          <th>Số cụm</th>
          <th>Điểm nhiễu</th>
          <th>ARI</th>
          <th>NMI</th>
          <th>Purity</th>
        </tr>
      </thead>
      <tbody>{''.join(rows)}</tbody>
    </table>
    """

def summary_cards_html(y, kmeans_labels, dbscan_labels, pca_explained):
    quality_counts = Counter(y)
    most_common_quality, most_common_count = quality_counts.most_common(1)[0]
    dbscan_noise = sum(1 for label in dbscan_labels if label == -1)
    pca_total = sum(pca_explained) * 100

    cards = [
        ("Số mẫu", f"{len(y)}", "Tổng số dòng dữ liệu hợp lệ được đọc từ file Excel."),
        ("Số nhãn quality", f"{len(quality_counts)}", f"Các nhãn: {sorted(quality_counts)}."),
        (
            "Nhãn phổ biến nhất",
            f"{most_common_quality}",
            f"{most_common_count} mẫu, chiếm {most_common_count * 100 / len(y):.1f}% dữ liệu.",
        ),
        (
            "PCA PC1 + PC2",
            f"{pca_total:.1f}%",
            "Tỉ lệ phương sai được biểu diễn trên mặt phẳng trực quan.",
        ),
        (
            "K-Means",
            f"{len(set(kmeans_labels))} cụm",
            "Số cụm k được đặt bằng số lượng nhãn quality khác nhau.",
        ),
        (
            "DBSCAN",
            f"{dbscan_noise} nhiễu",
            f"eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES}.",
        ),
    ]

    return "".join(
        f"""
        <article class="stat-card">
          <span>{escape(label)}</span>
          <strong>{escape(value)}</strong>
          <p>{escape(description)}</p>
        </article>
        """
        for label, value, description in cards
    )

def cluster_summary_table_html(title, y_true, y_pred):
    summary = cluster_quality_summary(y_true, y_pred)
    rows = []
    for cluster_label, values in summary.items():
        name = "Nhiễu" if cluster_label == -1 else str(cluster_label)
        rows.append(
            "<tr>"
            f"<th>{escape(name)}</th>"
            f"<td>{values['size']}</td>"
            f"<td>{values['main_quality']}</td>"
            f"<td>{values['main_count']}</td>"
            f"<td>{values['purity']:.3f}</td>"
            "</tr>"
        )

    return f"""
    <h2>{escape(title)}</h2>
    <div class="table-scroll">
      <table class="metrics-table">
        <thead>
          <tr>
            <th>Cụm</th>
            <th>Số mẫu</th>
            <th>Quality chiếm ưu thế</th>
            <th>Số mẫu của quality đó</th>
            <th>Độ tinh khiết cụm</th>
          </tr>
        </thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </div>
    """

def create_visual_report(
    feature_names,
    X,
    X_scaled,
    y,
    kmeans_labels,
    dbscan_labels,
    kmeans_metrics,
    dbscan_metrics,
    output_path=VISUAL_REPORT,
):
    projection, pca_explained = pca_2d(X_scaled)
    x_values = [point[0] for point in projection]
    y_values = [point[1] for point in projection]
    x_label = f"PC1 ({pca_explained[0] * 100:.1f}% phương sai)"
    y_label = f"PC2 ({pca_explained[1] * 100:.1f}% phương sai)"
    quality_counts = dict(sorted(Counter(y).items()))
    kmeans_counts = dict(sorted(Counter(kmeans_labels).items()))
    dbscan_counts = dict(sorted(Counter(dbscan_labels).items()))

    html = f"""<!doctype html>
<html lang="vi">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Trực quan gom cụm Wine Quality</title>
  <style>
    * {{
      box-sizing: border-box;
    }}
    body {{
      margin: 0;
      font-family: Arial, sans-serif;
      color: #111827;
      background: #f8fafc;
    }}
    main {{
      max-width: 1240px;
      margin: 0 auto;
      padding: 28px 20px 40px;
    }}
    h1 {{
      margin: 0 0 6px;
      font-size: 28px;
      line-height: 1.2;
    }}
    .summary {{
      max-width: 920px;
      margin: 0 0 22px;
      color: #4b5563;
      line-height: 1.6;
    }}
    .stats {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
      gap: 12px;
      margin: 20px 0;
    }}
    .stat-card {{
      background: #ffffff;
      border: 1px solid #e5e7eb;
      border-radius: 8px;
      padding: 14px;
      min-height: 122px;
      box-shadow: 0 1px 2px rgba(15, 23, 42, 0.06);
    }}
    .stat-card span {{
      display: block;
      color: #64748b;
      font-size: 12px;
      text-transform: uppercase;
      letter-spacing: 0.04em;
    }}
    .stat-card strong {{
      display: block;
      margin: 8px 0;
      font-size: 24px;
    }}
    .stat-card p {{
      margin: 0;
      color: #4b5563;
      font-size: 13px;
      line-height: 1.45;
    }}
    .grid {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(360px, 1fr));
      gap: 18px;
    }}
    .wide {{
      grid-column: 1 / -1;
    }}
    section {{
      background: white;
      border: 1px solid #e5e7eb;
      border-radius: 8px;
      padding: 14px;
      box-shadow: 0 1px 2px rgba(15, 23, 42, 0.06);
    }}
    h2 {{
      margin: 0 0 12px;
      font-size: 20px;
    }}
    p {{
      line-height: 1.6;
    }}
    ul {{
      margin: 0;
      padding-left: 20px;
      color: #374151;
      line-height: 1.65;
    }}
    svg {{
      display: block;
      width: 100%;
      height: auto;
    }}
    .chart-title {{
      font-size: 18px;
      font-weight: 700;
      fill: #111827;
    }}
    .axis {{
      stroke: #9ca3af;
      stroke-width: 1;
    }}
    .axis-label,
    .legend,
    .value {{
      font-size: 12px;
      fill: #4b5563;
    }}
    .metric-name {{
      font-size: 14px;
      font-weight: 700;
      fill: #111827;
    }}
    .table-scroll {{
      overflow-x: auto;
    }}
    table {{
      width: 100%;
      border-collapse: collapse;
      font-size: 14px;
    }}
    th,
    td {{
      border: 1px solid #e5e7eb;
      padding: 9px 10px;
      text-align: right;
      white-space: nowrap;
    }}
    th {{
      background: #f8fafc;
      text-align: left;
      font-weight: 700;
    }}
    .heatmap td {{
      text-align: center;
      font-weight: 700;
    }}
    .note {{
      color: #4b5563;
      margin-bottom: 0;
    }}
  </style>
</head>
<body>
  <main>
    <h1>Trực quan và đánh giá kết quả gom cụm Wine Quality</h1>
    <p class="summary">
      Báo cáo này được tạo bằng Python từ số liệu mới nhất sau khi chạy K-Means và DBSCAN.
      Cột nhãn <strong>quality</strong> không được đưa vào dữ liệu gom cụm; nhãn này chỉ được dùng
      để đối chiếu và tính các chỉ số ARI, NMI, Purity.
    </p>

    <div class="stats">
      {summary_cards_html(y, kmeans_labels, dbscan_labels, pca_explained)}
    </div>

    <div class="grid">
      <section class="wide">
        <h2>Mô tả cách trực quan</h2>
        <ul>
          <li>Dữ liệu gom cụm gồm {len(feature_names)} đặc trưng: {escape(', '.join(feature_names))}.</li>
          <li>Các đặc trưng được chuẩn hóa Z-score trước khi gom cụm để tránh đặc trưng có thang đo lớn chi phối khoảng cách.</li>
          <li>Ba biểu đồ phân tán bên dưới dùng PCA để chiếu {len(feature_names)} chiều về 2 chiều PC1 và PC2, giúp quan sát cấu trúc cụm tổng quát.</li>
          <li>Trong DBSCAN, cụm <strong>Nhiễu</strong> là các điểm không thuộc cụm mật độ nào.</li>
        </ul>
      </section>

      <section>{scatter_svg("Nhãn thật: quality trên mặt phẳng PCA", x_values, y_values, y, x_label, y_label)}</section>
      <section>{scatter_svg("Cụm K-Means trên mặt phẳng PCA", x_values, y_values, kmeans_labels, x_label, y_label)}</section>
      <section>{scatter_svg("Cụm DBSCAN trên mặt phẳng PCA", x_values, y_values, dbscan_labels, x_label, y_label)}</section>
      <section>{metrics_bar_svg(kmeans_metrics, dbscan_metrics)}</section>
      <section>{count_bar_svg("Phân bố nhãn quality", quality_counts, len(y))}</section>
      <section>{count_bar_svg("Kích thước cụm K-Means", kmeans_counts, len(y))}</section>
      <section>{count_bar_svg("Kích thước cụm DBSCAN", dbscan_counts, len(y))}</section>

      <section class="wide">
        <h2>Bảng chỉ số đánh giá</h2>
        {metric_table_html(kmeans_metrics, dbscan_metrics)}
        <p class="note">
          ARI và NMI càng gần 1 thì phân cụm càng trùng với nhãn quality. Purity càng cao thì mỗi cụm càng tập trung vào một nhãn quality.
        </p>
      </section>

      <section class="wide">
        {contingency_table_html("Bảng chéo quality và cụm K-Means", y, kmeans_labels)}
      </section>
      <section class="wide">
        {contingency_table_html("Bảng chéo quality và cụm DBSCAN", y, dbscan_labels)}
      </section>
      <section class="wide">
        {cluster_summary_table_html("Mô tả từng cụm K-Means", y, kmeans_labels)}
      </section>
      <section class="wide">
        {cluster_summary_table_html("Mô tả từng cụm DBSCAN", y, dbscan_labels)}
      </section>

      <section class="wide">
        <h2>Nhận xét ngắn</h2>
        <p>
          Kết quả cho thấy K-Means có ARI, NMI và Purity cao hơn DBSCAN với bộ tham số hiện tại.
          Tuy nhiên các chỉ số ARI/NMI vẫn thấp, nghĩa là cấu trúc cụm từ 11 đặc trưng hóa lý không trùng khớp mạnh với nhãn quality.
          Đây là tình huống thường gặp với bài toán gom cụm: nhãn đánh giá chất lượng là thông tin giám sát, còn gom cụm chỉ dựa trên độ gần nhau của đặc trưng.
        </p>
      </section>
    </div>
  </main>
</body>
</html>
"""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(html, encoding="utf-8")
    return output_path


## 9. Hàm chạy chính

**Công dụng:** Ghép tất cả bước lại với nhau: đọc dữ liệu, chuẩn hóa, chạy K-Means, chạy DBSCAN, đánh giá kết quả và tạo file `clustering_report.html`.


In [ ]:
print('Công dụng: Định nghĩa pipeline chính của bài toán.')

def main():
    feature_names, label_name, X, y = load_wine_quality(DATA_FILE)
    X_scaled = standardize(X)

    n_clusters = len(set(y))
    kmeans_labels = kmeans(X_scaled, n_clusters=n_clusters)
    dbscan_labels = dbscan(
        X_scaled,
        eps=DBSCAN_EPS,
        min_samples=DBSCAN_MIN_SAMPLES,
    )

    kmeans_metrics = evaluate_clustering(y, kmeans_labels)
    dbscan_metrics = evaluate_clustering(y, dbscan_labels)

    print(f"File du lieu: {DATA_FILE}")
    print(f"So mau: {len(X)}")
    print(f"So dac trung dung de gom cum: {len(feature_names)}")
    print(f"Cot nhan bi loai tru khoi X va chi dung de danh gia: {label_name}")
    print(f"Cac nhan chat luong: {sorted(set(y))}")
    print(f"K-Means k = so nhan khac nhau = {n_clusters}")
    print(f"DBSCAN eps = {DBSCAN_EPS}, min_samples = {DBSCAN_MIN_SAMPLES}")
    print()
    print_metrics("K-Means", kmeans_metrics)
    print()
    print_metrics("DBSCAN", dbscan_metrics)

    report_path = create_visual_report(
        feature_names,
        X,
        X_scaled,
        y,
        kmeans_labels,
        dbscan_labels,
        kmeans_metrics,
        dbscan_metrics,
    )
    print()
    print(f"Da tao bieu do truc quan: {report_path}")


## 10. Chạy toàn bộ chương trình

**Công dụng:** Gọi `main()` để in kết quả và tạo báo cáo trực quan. Sau khi chạy cell này, file `clustering_report.html` sẽ được cập nhật.


In [ ]:
print('Công dụng: Chạy toàn bộ pipeline và tạo báo cáo.')

main()
